<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 2 &middot; Part 4 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Feature Engineering, Polynomial Regression and scikit-learn

So far the features were handed to you. This lab is about *choosing* them: building new
features from raw measurements, fitting curves with a model that is still linear, and finally
checking everything you have written this week against a production library.

**Companion theory lecture:** CSE 815, Week 2, Part 2 (Feature Engineering and Polynomial Regression).

## What you will be able to do

1. Build an engineered feature from raw measurements and measure whether it earned its place.
2. Fit a curve with polynomial features, and explain why that is still linear regression.
3. Show, with numbers, why polynomial features make scaling mandatory rather than optional.
4. Choose between competing feature sets on evidence.
5. Reproduce your own results with `scikit-learn`, and use `StandardScaler` correctly.

---

## 1. Feature engineering: the area of a plot

Fifty plots of land in Chattogram. We measure **frontage** and **depth** in feet; the price
depends on neither directly &mdash; it depends on the **area**, which is their product.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)


def scale_zscore(X):
    """Z-score normalization; returns the scaled matrix and the statistics used."""
    mu = X.mean(axis=0)
    sigma = X.std(axis=0)
    return (X - mu) / sigma, mu, sigma


def compute_cost(X, y, w, b):
    err = X @ w + b - y
    return float(np.sum(err ** 2) / (2 * X.shape[0]))


def compute_gradient(X, y, w, b):
    m = X.shape[0]
    err = X @ w + b - y
    return X.T @ err / m, float(np.sum(err) / m)


def gradient_descent(X, y, alpha, num_iters, w_init=None, b_init=0.0, record_every=1):
    """Batch gradient descent; carried over unchanged from Parts 2 and 3."""
    w = np.zeros(X.shape[1]) if w_init is None else np.array(w_init, dtype=float)
    b = float(b_init)
    history = {"cost": []}
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if i % record_every == 0 or i == num_iters - 1:
            history["cost"].append(compute_cost(X, y, w, b))
    return w, b, history


def fit_exact(X, y):
    """Closed-form least squares. Returns w, b and the minimum cost."""
    A = np.column_stack([X, np.ones(X.shape[0])])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return coef[:-1], float(coef[-1]), compute_cost(X, y, coef[:-1], coef[-1])


def r_squared(y, y_hat):
    """Fraction of the variance in y explained by the predictions."""
    return 1.0 - np.sum((y - y_hat) ** 2) / np.sum((y - y.mean()) ** 2)


def load_plots(seed=2025, m=50):
    """Plots of land: frontage and depth in feet, price in lakh BDT."""
    rng = np.random.default_rng(seed)
    frontage = np.round(rng.uniform(15, 70, m), 0)
    depth    = np.round(rng.uniform(30, 140, m), 0)
    price    = 0.0042 * frontage * depth + rng.normal(0, 0.8, m) + 5.0
    return np.column_stack([frontage, depth]), np.round(price, 1)


X_plot, y_plot = load_plots()
print("X_plot.shape:", X_plot.shape)
print(f"\n{'frontage':>9} {'depth':>7} {'area':>8} {'price':>8}")
print("-" * 36)
for i in range(6):
    f, d = X_plot[i]
    print(f"{f:9.0f} {d:7.0f} {f*d:8.0f} {y_plot[i]:8.1f}")

In [ ]:
w_raw, b_raw, J_raw = fit_exact(X_plot, y_plot)
print("frontage and depth only")
print(f"  w = {w_raw},  b = {b_raw:.4f}")
print(f"  J = {J_raw:.4f}   R^2 = {r_squared(y_plot, X_plot @ w_raw + b_raw):.4f}")

$R^2$ of about $0.94$ looks respectable. But the model
$f = w_1 \cdot \text{frontage} + w_2 \cdot \text{depth} + b$ treats the two measurements as
*separate additive contributions*, and that is the wrong shape for a product. The evidence is
in the residuals: plot them against area and a pattern appears that a correct model would not
leave behind.

In [ ]:
area = X_plot[:, 0] * X_plot[:, 1]
resid_raw = (X_plot @ w_raw + b_raw) - y_plot

plt.figure(figsize=(6.5, 4))
plt.scatter(area, resid_raw, s=32, c="#9B2C4B")
plt.axhline(0, c="#4A5560", lw=1.2)
plt.xlabel("area (sq ft)")
plt.ylabel("residual (lakh BDT)")
plt.title("Residuals of the frontage + depth model")
plt.show()

print("A well-specified model leaves residuals with no structure. These have a clear curve:")
print("  small plots are over-predicted, mid-sized ones under-predicted.")

### Add the feature the domain suggests

$$x_3 = x_1 x_2 = \text{area}$$

The model can now express something no combination of $w_1$ and $w_2$ could.

In [ ]:
X_area = np.column_stack([X_plot, X_plot[:, 0] * X_plot[:, 1]])
w_eng, b_eng, J_eng = fit_exact(X_area, y_plot)

print("frontage, depth AND area")
print(f"  w = {w_eng},  b = {b_eng:.4f}")
print(f"  J = {J_eng:.4f}   R^2 = {r_squared(y_plot, X_area @ w_eng + b_eng):.4f}")
print()
print(f"cost fell from {J_raw:.4f} to {J_eng:.4f}  --  a factor of {J_raw / J_eng:.1f}")
print(f"the fitted weight on area is {w_eng[2]:.6f}; the rule that generated the data used 0.004200")

assert J_eng < J_raw / 3, "the engineered feature should cut the cost substantially"

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4), sharey=True)
for a, (resid, title) in zip(ax, [(resid_raw, "frontage + depth"),
                                  ((X_area @ w_eng + b_eng) - y_plot, "+ area")]):
    a.scatter(area, resid, s=32, c="#9B2C4B")
    a.axhline(0, c="#4A5560", lw=1.2)
    a.set_xlabel("area (sq ft)")
    a.set_title(title)
ax[0].set_ylabel("residual (lakh BDT)")
plt.tight_layout()
plt.show()

The structure is gone: what is left looks like the measurement noise the data was built with.
That is what "the feature earned its place" looks like.

> **Feature engineering is a hypothesis about the world.** *Area drives price* is a claim you
> can state before fitting anything, and the residual plot is how you test it. Adding
> `frontage * depth` because it happened to help would be a different, much weaker activity.

---

## 2. Polynomial regression

A second data set: flat price against size alone, where price rises steeply at first and then
**flattens**. A straight line cannot bend.

In [ ]:
def load_sizes(seed=316, m=40):
    """Flat price against size only, with a relationship that flattens."""
    rng = np.random.default_rng(seed)
    x = np.round(np.sort(rng.uniform(0.5, 3.5, m)), 2)
    y = np.round(46 * np.sqrt(x) - 8 + rng.normal(0, 2.2, m), 1)
    return x, y


x_size, y_size = load_sizes()
print(f"m = {x_size.shape[0]},  x in [{x_size.min():.2f}, {x_size.max():.2f}],"
      f"  y in [{y_size.min():.1f}, {y_size.max():.1f}]")

plt.figure(figsize=(6.5, 4))
plt.scatter(x_size, y_size, s=35, c="#1B7F79")
plt.xlabel("size (1000 sq ft)")
plt.ylabel("price (lakh BDT)")
plt.title("Price rises, then flattens")
plt.show()

Feed the model **powers of $x$** as extra features:

$$f(x) = w_1 x + w_2 x^2 + w_3 x^3 + b$$

This is still **linear regression** &mdash; linear in the *parameters*. Only the feature matrix
changed. Nothing in `compute_cost`, `compute_gradient` or `gradient_descent` is touched.

In [ ]:
def poly_features(x, degree):
    """Column stack of x, x^2, ..., x^degree."""
    return np.column_stack([x ** p for p in range(1, degree + 1)])


candidates = {
    "x":              poly_features(x_size, 1),
    "x, x^2":         poly_features(x_size, 2),
    "x, x^2, x^3":    poly_features(x_size, 3),
    "sqrt(x)":        np.column_stack([np.sqrt(x_size)]),
}

print(f"{'features':14s} {'n':>3} {'J_train':>10} {'R^2':>9}   weights")
print("-" * 74)
fits = {}
for label, F in candidates.items():
    w, b, J = fit_exact(F, y_size)
    fits[label] = (F, w, b, J)
    print(f"{label:14s} {F.shape[1]:>3} {J:10.4f} {r_squared(y_size, F @ w + b):9.4f}   "
          f"w={np.round(w, 3)}  b={b:.3f}")

In [ ]:
x_line = np.linspace(x_size.min(), x_size.max(), 200)
line_features = {
    "x":           poly_features(x_line, 1),
    "x, x^2":      poly_features(x_line, 2),
    "x, x^2, x^3": poly_features(x_line, 3),
    "sqrt(x)":     np.column_stack([np.sqrt(x_line)]),
}
colours = {"x": "#9B2C4B", "x, x^2": "#1B7F79", "x, x^2, x^3": "#C97B17", "sqrt(x)": "#123B63"}

plt.figure(figsize=(7, 4.6))
plt.scatter(x_size, y_size, s=35, c="#4A5560", alpha=0.7, label="data", zorder=3)
for label, (F, w, b, J) in fits.items():
    style = "--" if label == "sqrt(x)" else "-"
    plt.plot(x_line, line_features[label] @ w + b, style, lw=2,
             c=colours[label], label=f"{label}  (J = {J:.2f})")
plt.xlabel("size (1000 sq ft)")
plt.ylabel("price (lakh BDT)")
plt.title("Four feature sets, one linear regression")
plt.legend(fontsize=8)
plt.show()

Read the table and the plot together. `sqrt(x)` uses **one** feature and reaches a cost of
$2.27$; `x, x^2, x^3` uses **three** and reaches $2.17$. Three features buy almost nothing
over the single right one.

That is the lecture's point about functional form. A quadratic eventually turns downward, so
it predicts that very large flats get cheaper. $\sqrt{x}$ rises forever, ever more slowly,
which is the behaviour we actually believe. Look at what happens outside the training range.

In [ ]:
x_far = np.linspace(0.5, 8.0, 200)

plt.figure(figsize=(7, 4.4))
plt.scatter(x_size, y_size, s=30, c="#4A5560", alpha=0.7, zorder=3, label="training range")
for label in ["x, x^2", "x, x^2, x^3", "sqrt(x)"]:
    _, w, b, _ = fits[label]
    F_far = (np.column_stack([np.sqrt(x_far)]) if label == "sqrt(x)"
             else poly_features(x_far, len(w)))
    plt.plot(x_far, F_far @ w + b, lw=2, c=colours[label], label=label)
plt.axvspan(x_size.min(), x_size.max(), color="#EEF3F7", zorder=0)
plt.ylim(-40, 140)
plt.xlabel("size (1000 sq ft)")
plt.ylabel("price (lakh BDT)")
plt.title("Extrapolating beyond the data (shaded = training range)")
plt.legend(fontsize=8)
plt.show()

Inside the shaded band all three agree. Outside it the quadratic turns over and eventually
predicts negative prices, the cubic runs away upward, and only $\sqrt{x}$ keeps behaving. All
three fit the training data about equally well; they disagree completely about everything
else.

> The tools for choosing between models on evidence rather than on training cost &mdash;
> held-out data, bias and variance, regularization &mdash; arrive in Week 3 and Module 2. For
> now, domain knowledge and a look at the extrapolation are what you have, and they are worth
> more than an extra decimal place of $J_{\text{train}}$.

---

## 3. Polynomial features make scaling mandatory

If $x \in [0.66, 3.49]$ then $x^3 \in [0.29, 42.5]$. Powers explode the range, and you already
know from Part 3 exactly what that does.

In [ ]:
F3 = poly_features(x_size, 3)
print(f"{'feature':>10} {'min':>10} {'max':>10} {'std':>10}")
print("-" * 44)
for j, name in enumerate(["x", "x^2", "x^3"]):
    print(f"{name:>10} {F3[:, j].min():10.3f} {F3[:, j].max():10.3f} {F3[:, j].std():10.3f}")


def curvature(X):
    A = np.column_stack([X, np.ones(X.shape[0])])
    ev = np.linalg.eigvalsh(A.T @ A / X.shape[0])
    return 2.0 / ev.max(), ev.max() / ev.min()


F3_s, mu3, sigma3 = scale_zscore(F3)
a_raw, k_raw = curvature(F3)
a_sc, k_sc = curvature(F3_s)

print(f"\n{'':10s} {'alpha_max':>12} {'kappa':>14}")
print("-" * 38)
print(f"{'raw':10s} {a_raw:12.3e} {k_raw:14.3e}")
print(f"{'z-scored':10s} {a_sc:12.3e} {k_sc:14.3e}")
print(f"\nScaling raises the usable alpha by a factor of {a_sc / a_raw:,.0f}.")

In [ ]:
import warnings

_, _, J3 = fit_exact(F3, y_size)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    _, _, h_raw = gradient_descent(F3, y_size, alpha=a_raw / 2, num_iters=3000)
w_s, b_s, h_sc = gradient_descent(F3_s, y_size, alpha=0.3, num_iters=3000)

print(f"minimum cost                              : {J3:.4f}")
print(f"raw features,  alpha = {a_raw/2:.2e}, 3000 iters : {h_raw['cost'][-1]:.4f}")
print(f"z-scored,      alpha = 0.30, 3000 iters       : {h_sc['cost'][-1]:.4f}")

plt.figure(figsize=(6.5, 4))
plt.plot(h_raw["cost"], c="#9B2C4B", lw=2, label=f"raw powers, alpha = {a_raw/2:.1e}")
plt.plot(h_sc["cost"], c="#1B7F79", lw=2, label="z-scored, alpha = 0.3")
plt.axhline(J3, ls="--", c="#C97B17", lw=1.4, label=f"minimum J = {J3:.2f}")
plt.yscale("log")
plt.xlabel("iteration"); plt.ylabel("J   (log scale)")
plt.title("Polynomial features, with and without scaling")
plt.legend(fontsize=8)
plt.show()

assert h_sc["cost"][-1] < h_raw["cost"][-1]

Even given a learning rate chosen to be exactly half of its stability limit, the unscaled run
is nowhere near the minimum after 3000 iterations. The scaled run is there almost immediately.

**Rule:** the moment you add a polynomial feature, scale.

### Checkpoint 1

For the plots data set in section 1, the engineered `area` feature has a range of roughly
$[760, 8900]$ while frontage spans $[15, 70]$. Z-score `X_area`, run `gradient_descent` with
`alpha = 0.3` for 2000 iterations, and confirm the cost reaches the closed-form minimum
`J_eng`.

Then unscale the weights back to raw units, using
$w_j^{\text{raw}} = w_j/\sigma_j$ and $b^{\text{raw}} = b - \sum_j w_j\mu_j/\sigma_j$, and
check they match `w_eng`, `b_eng`.

*Expected:* a final cost within `1e-6` of `J_eng`, and weights matching `w_eng` to at least
four decimal places.

In [ ]:
# Your code here

---

## 4. The same thing with scikit-learn

Everything you wrote this week exists in `scikit-learn`, tested and optimised. You wrote it
yourself so that you know what the library is doing &mdash; and so that you can tell when it is
doing the wrong thing. From here on, use the library and check it against your understanding.

In [ ]:
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

FEATURES = ["size (k sq ft)", "bedrooms", "floor", "age (yr)"]


def load_flats(seed=815, m=60):
    """The Week 2 flats data set, as in Parts 2 and 3."""
    rng = np.random.default_rng(seed)
    size  = np.round(rng.uniform(0.60, 3.00, m), 2)
    beds  = np.clip(np.round(1.2 + 1.7 * size + rng.normal(0, 0.6, m)), 1, 6)
    floor = rng.integers(1, 13, m).astype(float)
    age   = np.round(rng.uniform(1, 60, m), 0)
    price = (22.0 * size + 2.5 * beds + 0.9 * floor - 0.35 * age + 8.0
             + rng.normal(0, 2.5, m))
    return np.column_stack([size, beds, floor, age]), np.round(price, 1)


X_flats, y_flats = load_flats()
w_ref, b_ref, J_ref = fit_exact(X_flats, y_flats)
print("your closed-form fit:  w =", w_ref, " b = %.4f" % b_ref)

### `LinearRegression` &mdash; the normal equation

`LinearRegression` solves the least-squares problem directly, exactly as `np.linalg.lstsq`
does. No learning rate, no iterations, no scaling required.

In [ ]:
lin = LinearRegression().fit(X_flats, y_flats)

print(f"{'feature':16s} {'yours':>12} {'sklearn':>12}")
print("-" * 42)
for j, name in enumerate(FEATURES):
    print(f"{name:16s} {w_ref[j]:12.4f} {lin.coef_[j]:12.4f}")
print(f"{'bias':16s} {b_ref:12.4f} {lin.intercept_:12.4f}")
print(f"\nR^2 = {lin.score(X_flats, y_flats):.6f}")

assert np.allclose(lin.coef_, w_ref)
assert np.isclose(lin.intercept_, b_ref)
print("\nIdentical -- your implementation and the library solve the same problem.")

### `StandardScaler` &mdash; your `scale_zscore`, with the statistics remembered

`StandardScaler` is the packaged version of Part 3's rule. `fit` learns $\mu$ and $\sigma$;
`transform` applies them. Calling `fit` on your test data is the bug from Part 3, and the
API's shape is designed to make it obvious.

In [ ]:
scaler = StandardScaler().fit(X_flats)
X_flats_s = scaler.transform(X_flats)

_, mu_mine, sigma_mine = scale_zscore(X_flats)
print("mean_  :", scaler.mean_)
print("mine   :", mu_mine)
print("scale_ :", scaler.scale_)
print("mine   :", sigma_mine)

assert np.allclose(scaler.mean_, mu_mine)
assert np.allclose(scaler.scale_, sigma_mine)
print("\nStandardScaler is exactly the z-score you implemented.")

### `SGDRegressor` &mdash; and why it needs the scaler

`SGDRegressor` is gradient descent. Give it the raw features and it does exactly what your
own code did in Part 2, only worse: this is a library, not a safety net.

In [ ]:
sgd_scaled = SGDRegressor(max_iter=5000, tol=1e-7, random_state=0)
sgd_scaled.fit(X_flats_s, y_flats)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sgd_raw = SGDRegressor(max_iter=5000, tol=1e-7, random_state=0)
    sgd_raw.fit(X_flats, y_flats)

w_ref_s, b_ref_s, _ = fit_exact(X_flats_s, y_flats)

print("on SCALED features")
print("  SGDRegressor  w =", np.round(sgd_scaled.coef_, 3), " b = %.3f" % sgd_scaled.intercept_[0])
print("  exact         w =", np.round(w_ref_s, 3), " b = %.3f" % b_ref_s)
print()
print("on RAW features")
print("  SGDRegressor  w =", sgd_raw.coef_)
print("  exact         w =", np.round(w_ref, 3))
print(f"\n  largest coefficient magnitude: scaled {np.abs(sgd_scaled.coef_).max():.2f}"
      f"   raw {np.abs(sgd_raw.coef_).max():.3e}")

assert np.allclose(sgd_scaled.coef_, w_ref_s, atol=1.0), "scaled SGD should land near the minimum"
assert np.abs(sgd_raw.coef_).max() > 1e3, "raw SGD should blow up"
print("\nThe scaled run found the minimum. The raw run diverged, silently, with no exception.")

> That is the single most important thing to take from this section. `sgd_raw.fit(...)`
> returned normally. `sgd_raw.predict(...)` will return numbers. Nothing in the API tells you
> the model is garbage &mdash; only knowing *why* scaling matters does.

### A pipeline keeps the scaler attached to the model

A model without its scaler is not a model. `make_pipeline` bundles them so that `fit` fits
the scaler on training data only, and `predict` applies the stored statistics.

In [ ]:
pipe = make_pipeline(StandardScaler(), SGDRegressor(max_iter=5000, tol=1e-7, random_state=0))
pipe.fit(X_flats, y_flats)

x_new = np.array([[1.80, 3.0, 5.0, 12.0]])          # size, bedrooms, floor, age

print(f"pipeline prediction        : {pipe.predict(x_new)[0]:.2f} lakh BDT")
print(f"LinearRegression prediction: {lin.predict(x_new)[0]:.2f} lakh BDT")
print(f"your closed-form model     : {(x_new[0] @ w_ref + b_ref):.2f} lakh BDT")

print(f"\npipeline R^2 = {pipe.score(X_flats, y_flats):.6f}")
assert abs(pipe.predict(x_new)[0] - (x_new[0] @ w_ref + b_ref)) < 1.0

### Checkpoint 2

`sklearn.preprocessing.PolynomialFeatures(degree=3, include_bias=False)` generates the
polynomial features of section 2 for you. Build a pipeline

`make_pipeline(PolynomialFeatures(3, include_bias=False), StandardScaler(), LinearRegression())`

fit it on `x_size.reshape(-1, 1)` and `y_size`, and compare its $R^2$ with the value you got
for `x, x^2, x^3` in section 2.

*Expected:* $R^2 \approx 0.9791$, matching your own fit &mdash; scaling changes the path to the
minimum, never the minimum itself.

In [ ]:
# Your code here

---

## 5. Recap of Week 2

| Idea | Where | Code |
|---|---|---|
| feature matrix | Part 2 | `X` of shape `(m, n)` |
| model | Part 2 | `X @ w + b` |
| gradient | Part 2 | `X.T @ err / m`, `err.mean()` |
| z-scoring | Part 3 | `(X - X.mean(0)) / X.std(0)` |
| choosing $\alpha$ | Part 3 | $\times 3$ grid, largest that decreases |
| engineered feature | Part 4 | `np.column_stack([X, X[:,0] * X[:,1]])` |
| polynomial features | Part 4 | `np.column_stack([x**p for p in ...])` |
| the library | Part 4 | `make_pipeline(StandardScaler(), SGDRegressor())` |

- An engineered feature is a hypothesis. Test it with a residual plot, not with $J$ alone.
- Polynomial regression is linear regression on transformed features. Nothing in the
  optimiser changes.
- The right functional form beats more powers: one $\sqrt{x}$ feature matched three
  polynomial ones here, and behaves outside the data.
- Powers explode feature ranges, so scaling stops being optional.
- `LinearRegression` is the closed form; `SGDRegressor` is gradient descent and needs scaled
  features; a `Pipeline` keeps the scaler with the model.

### Exercises to hand in

1. For the plots data, fit a model on `area` **alone** (one feature). Compare its $J$ and
   $R^2$ with the three-feature model of section 1. Which would you deploy, and why? What
   does the answer say about keeping frontage and depth once area is available?
2. Fit degrees 1 through 9 to `x_size, y_size` using `fit_exact` and plot $J_{\text{train}}$
   against degree. Describe the curve, then explain in three or four sentences why you cannot
   choose the degree from this plot alone. Name the tool from Week 3 that you would need.
3. Take the flats data of Parts 2 and 3 and engineer two new features: `size / bedrooms`
   (average room size) and `size * floor`. Report the change in $J_{\min}$ for each, and
   state whether either is worth keeping and on what evidence.
4. Reproduce section 3's comparison for degree 6 instead of degree 3. Report `alpha_max` for
   the raw and scaled feature matrices, and the ratio between them.
5. Deliberately misuse the pipeline: fit a `StandardScaler` on the whole flats data set, then
   split into train and held-out sets and evaluate. Compare with the correct order (split
   first, fit the scaler on the training part only). Report both held-out costs and explain
   which number you would be entitled to publish.
6. **Practice lab (assessed).** Using only the tools from Week 2, build the best model you can
   for `load_flats(seed=99, m=200)`: choose the features, scale them, choose $\alpha$, train
   with your own `gradient_descent`, and confirm the result against `LinearRegression`. Hand
   in the notebook plus half a page justifying each choice.

### Next

**Week 3 &mdash; Classification:** the target becomes a category rather than a number. The
logistic model, decision boundaries, a cost function that squared error cannot provide, and
the problem of overfitting.

**Reading:** James et al., *ISL* 2e, &sect;3.3 (interaction terms and non-linear
transformations of the predictors); Bishop, *PRML*, &sect;1.1 (polynomial curve fitting).